In [ ]:
import json
import re
import csv
import glob
import os
from fractions import Fraction
from typing import Optional

class MathVistaNormalizer:
    @classmethod
    def to_numeric(cls, raw: str) -> Optional[float]:
        if not raw: return None
        raw = str(raw).replace('$', '').replace(' ', '').lower().strip()
        raw = re.sub(r"(?:cm|m/s|kg|ms|n|grams|degrees|°|%)", "", raw)
        
        # 1. 根式
        sqrt_match = re.search(r"(?:√|\\sqrt\s*\{)\s*(\d+)\s*\}?", raw)
        if sqrt_match: return float(int(sqrt_match.group(1))**0.5)

        # 2. 百分数
        if '%' in raw or r'\%' in raw:
            m = re.search(r"(-?\d+(?:\.\d+)?)", raw)
            if m: return float(m.group(1)) / 100.0

        # 3. 分数
        if '/' in raw:
            m = re.search(r"(-?\d+)\s*/\s*(-?\d+)", raw)
            if m:
                try: return float(Fraction(int(m.group(1)), int(m.group(2))))
                except: pass
            
        # 4. 普通数值提取
        nums = re.findall(r"[-+]?\d*\.?\d+", raw.replace(',', ''))
        if nums:
            try: return float(nums[-1])
            except: return None
        return None

def full_post_process_and_save(results_path, eval_data_path):
    # 加载原始数据索引
    with open(eval_data_path, 'r') as f:
        eval_ref = {str(item['pid']): item for item in json.load(f)}
    
    with open(results_path, 'r') as f:
        data = json.load(f)
    
    records = data['records']
    new_correct = 0
    fix_count = 0
    
    # --- [关键修改]: 定义 log_buffer 和 csv_data ---
    log_buffer = []
    csv_data = []

    for rec in records:
        pid = str(rec['pid'])
        ref = eval_ref.get(pid, {})
        choices = ref.get('choices', [])
        
        ext = str(rec['ext_p']).strip()
        gt = str(rec['gt']).strip()
        is_now_correct = False

        # 1. 选项映射
        if choices and len(ext) == 1 and ext.upper() in "ABCDE":
            idx = ord(ext.upper()) - ord('A')
            if idx < len(choices): ext = str(choices[idx]).strip()

        # 2. 数值对比
        val_p = MathVistaNormalizer.to_numeric(ext)
        val_g = MathVistaNormalizer.to_numeric(gt)
        if val_p is not None and val_g is not None:
            if abs(val_p - val_g) < 1e-4: is_now_correct = True

        # 3. 语义对齐
        if not is_now_correct:
            ext_l = ext.lower().strip().rstrip('.')
            gt_l = gt.lower().strip().rstrip('.')
            bool_map = {"yes": "true", "no": "false", "correct": "true", "incorrect": "false"}
            if bool_map.get(ext_l, ext_l) == bool_map.get(gt_l, gt_l): is_now_correct = True

        # 更新计数
        if is_now_correct:
            new_correct += 1
            if not rec.get('is_correct', False): fix_count += 1
        
        status = "✅" if is_now_correct else "❌"
        if is_now_correct and not rec.get('is_correct', False): status = "⭐ FIX"
        
        # 存入 buffer
        line = f"{pid:<6} | {ext[:25]:<25} | {gt[:20]:<20} | {status}"
        log_buffer.append(line)
        
        # 存入 csv_data 用于后续分析
        csv_data.append([pid, ext, gt, status, rec.get('is_correct', False), is_now_correct])

    total = len(records)
    final_acc = new_correct / total
    
    # --- 1. 打印全部 (Jupyter 会截断，但我们可以看最后几行) ---
    print("\n".join(log_buffer)) 

    print("-" * 85)
    print(f"统计汇总: 样本数 {total} | 原始 Acc: {data['eval_result'].get('accuracy', 0):.2%} | 后处理后 Acc: {final_acc:.2%}")
    print(f"提升点数: {(final_acc - data['eval_result'].get('accuracy', 0))*100:.2f} pts | 救回样本: {fix_count}")

    # --- 2. 导出 CSV (推荐用这个看全部 500 条) ---
    csv_name = os.path.basename(results_path).replace(".json", "_analysis.csv")
    with open(csv_name, "w", newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(["PID", "Final_Extracted", "Ground_Truth", "Status", "Original_Correct", "New_Correct"])
        writer.writerows(csv_data)
    
    # print("\n✅ 所有 500 条数据已保存至: mathvista_full_analysis.csv")
    print(f"\n✅ 所有 500 条数据已保存至: {csv_name}")
    return final_acc, data['eval_result'].get('accuracy', 0)


# 运行
# full_post_process_and_save(
#     results_path = "/data1/wzy/cot-mimic/results/record/icl-qwen2.5-vl-7b-instruct-mathvista-0shot-direct/0shot_5_baseline.json",
#     eval_data_path = "/data1/wzy/cot-mimic/mathvista/data/mathvista_evaluate.json" 
# )

results_dir = "/data1/wzy/cot-mimic/results/record/licv_inject_gsm8k"
eval_data_path = "/data1/wzy/cot-mimic/mathvista/data/mathvista_evaluate.json"

result_files = sorted(glob.glob(os.path.join(results_dir, "licv_layers_*_direct_q_1.0.json")))

print(f"发现 {len(result_files)} 个结果文件\n")
summary = []


summary = []

for path in result_files:
    print("=" * 100)
    print(f"正在处理: {os.path.basename(path)}")
    print("=" * 100)

    # ✅ 接住返回值
    final_acc, orig_acc = full_post_process_and_save(
        results_path=path,
        eval_data_path=eval_data_path
    )

    # ✅ 保存到 summary
    summary.append((os.path.basename(path), orig_acc, final_acc))


# ================= 汇总打印（放在循环外面） =================

print("\n" + "=" * 90)
print(f"{'File':40} | {'Orig Acc':9} | {'Post Acc':9} | {'Gain'}")
print("-" * 90)

best_file, best_acc = None, 0

for name, orig, post in summary:
    gain = (post - orig) * 100
    print(f"{name:40} | {orig*100:8.2f}% | {post*100:8.2f}% | {gain:+.2f}")

    if post > best_acc:
        best_acc = post
        best_file = name

print("-" * 90)
print(f"✅ Best Layer: {best_file} -> {best_acc*100:.2f}%")
print("=" * 90)



发现 28 个结果文件

正在处理: licv_layers_0_direct_q_1.0.json
506    | 2014 and 2015             | 2016                 | ❌
901    | 7                         | 7                    | ✅
945    | The image provided does n | 6                    | ❌
482    | 45                        | 60                   | ❌
307    | -3                        | 2.58                 | ❌
249    | 6                         | 6                    | ✅
407    | 40                        | 40                   | ✅
952    | Fish                      | Fish                 | ⭐ FIX
207    | 5                         | 5                    | ✅
358    | E                         | A                    | ❌
208    | The image provided does n | 5                    | ❌
798    | 180                       | k + p + s            | ❌
279    | grass                     | grass                | ⭐ FIX
113    | 24                        | 20                   | ❌
946    | The image provided is a p | 16                   | ❌
716    | 16

In [ ]:
import json
import re
import csv
import glob
import os
from fractions import Fraction
from typing import Optional

class MathVistaNormalizer:
    @classmethod
    def to_numeric(cls, raw: str) -> Optional[float]:
        if not raw: return None
        raw = str(raw).replace('$', '').replace(' ', '').lower().strip()
        raw = re.sub(r"(?:cm|m/s|kg|ms|n|grams|degrees|°|%)", "", raw)
        
        # 1. 根式
        sqrt_match = re.search(r"(?:√|\\sqrt\s*\{)\s*(\d+)\s*\}?", raw)
        if sqrt_match: return float(int(sqrt_match.group(1))**0.5)

        # 2. 百分数
        if '%' in raw or r'\%' in raw:
            m = re.search(r"(-?\d+(?:\.\d+)?)", raw)
            if m: return float(m.group(1)) / 100.0

        # 3. 分数
        if '/' in raw:
            m = re.search(r"(-?\d+)\s*/\s*(-?\d+)", raw)
            if m:
                try: return float(Fraction(int(m.group(1)), int(m.group(2))))
                except: pass
            
        # 4. 普通数值提取
        nums = re.findall(r"[-+]?\d*\.?\d+", raw.replace(',', ''))
        if nums:
            try: return float(nums[-1])
            except: return None
        return None

def full_post_process_and_save(results_path, eval_data_path):
    # 加载原始数据索引
    with open(eval_data_path, 'r') as f:
        eval_ref = {str(item['pid']): item for item in json.load(f)}
    
    with open(results_path, 'r') as f:
        data = json.load(f)
    
    records = data['records']
    new_correct = 0
    fix_count = 0
    
    # --- [关键修改]: 定义 log_buffer 和 csv_data ---
    log_buffer = []
    csv_data = []

    for rec in records:
        pid = str(rec['pid'])
        ref = eval_ref.get(pid, {})
        choices = ref.get('choices', [])
        
        ext = str(rec['ext_p']).strip()
        gt = str(rec['gt']).strip()
        is_now_correct = False

        # 1. 选项映射
        if choices and len(ext) == 1 and ext.upper() in "ABCDE":
            idx = ord(ext.upper()) - ord('A')
            if idx < len(choices): ext = str(choices[idx]).strip()

        # 2. 数值对比
        val_p = MathVistaNormalizer.to_numeric(ext)
        val_g = MathVistaNormalizer.to_numeric(gt)
        if val_p is not None and val_g is not None:
            if abs(val_p - val_g) < 1e-4: is_now_correct = True

        # 3. 语义对齐
        if not is_now_correct:
            ext_l = ext.lower().strip().rstrip('.')
            gt_l = gt.lower().strip().rstrip('.')
            bool_map = {"yes": "true", "no": "false", "correct": "true", "incorrect": "false"}
            if bool_map.get(ext_l, ext_l) == bool_map.get(gt_l, gt_l): is_now_correct = True

        # 更新计数
        if is_now_correct:
            new_correct += 1
            if not rec.get('is_correct', False): fix_count += 1
        
        status = "✅" if is_now_correct else "❌"
        if is_now_correct and not rec.get('is_correct', False): status = "⭐ FIX"
        
        # 存入 buffer
        line = f"{pid:<6} | {ext[:25]:<25} | {gt[:20]:<20} | {status}"
        log_buffer.append(line)
        
        # 存入 csv_data 用于后续分析
        csv_data.append([pid, ext, gt, status, rec.get('is_correct', False), is_now_correct])

    total = len(records)
    final_acc = new_correct / total
    
    # --- 1. 打印全部 (Jupyter 会截断，但我们可以看最后几行) ---
    # print("\n".join(log_buffer)) 

    # print("-" * 85)
    print(f"统计汇总: 样本数 {total} | 原始 Acc: {data['eval_result'].get('accuracy', 0):.2%} | 后处理后 Acc: {final_acc:.2%}")
    print(f"提升点数: {(final_acc - data['eval_result'].get('accuracy', 0))*100:.2f} pts | 救回样本: {fix_count}")

    # --- 2. 导出 CSV (推荐用这个看全部 500 条) ---
    csv_name = os.path.basename(results_path).replace(".json", "_analysis.csv")
    with open(csv_name, "w", newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(["PID", "Final_Extracted", "Ground_Truth", "Status", "Original_Correct", "New_Correct"])
        writer.writerows(csv_data)
    
    # print("\n✅ 所有 500 条数据已保存至: mathvista_full_analysis.csv")
    # print(f"\n✅ 所有 500 条数据已保存至: {csv_name}")
    return final_acc, data['eval_result'].get('accuracy', 0)

results_dir = "/data1/wzy/cot-mimic/results/record/licv_inject_gsm8k"
eval_data_path = "/data1/wzy/cot-mimic/mathvista/data/mathvista_evaluate.json"

# result_files = sorted(glob.glob(os.path.join(results_dir, "licv_layers_*_direct_q_1.0.json")))
result_files = glob.glob(os.path.join(results_dir, "licv_layers_*_direct_q_1.0.json"))

result_files = sorted(result_files, key=lambda x: int(re.search(r"layers_(\d+)", x).group(1)))


# print(f"发现 {len(result_files)} 个结果文件\n")
summary = []

for path in result_files:
    print("=" * 100)
    print(f"正在处理: {os.path.basename(path)}")
    print("=" * 100)

    # ✅ 接住返回值
    final_acc, orig_acc = full_post_process_and_save(
        results_path=path,
        eval_data_path=eval_data_path
    )

    # ✅ 保存到 summary
    summary.append((os.path.basename(path), orig_acc, final_acc))


# ================= 汇总打印（放在循环外面） =================

# print("\n" + "=" * 90)
# print(f"{'File':40} | {'Orig Acc':9} | {'Post Acc':9} | {'Gain'}")
# print("-" * 90)

best_file, best_acc = None, 0

for name, orig, post in summary:
    gain = (post - orig) * 100
    print(f"{name:40} | {orig*100:8.2f}% | {post*100:8.2f}% | {gain:+.2f}")

    if post > best_acc:
        best_acc = post
        best_file = name

print("-" * 90)
print(f"✅ Best Layer: {best_file} -> {best_acc*100:.2f}%")
print("=" * 90)



正在处理: licv_layers_0_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 29.80% | 后处理后 Acc: 55.00%
提升点数: 25.20 pts | 救回样本: 126
正在处理: licv_layers_10_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.60% | 后处理后 Acc: 56.00%
提升点数: 25.40 pts | 救回样本: 127
正在处理: licv_layers_11_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 33.80% | 后处理后 Acc: 54.80%
提升点数: 21.00 pts | 救回样本: 105
正在处理: licv_layers_12_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 31.80% | 后处理后 Acc: 59.00%
提升点数: 27.20 pts | 救回样本: 136
正在处理: licv_layers_13_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 31.20% | 后处理后 Acc: 51.60%
提升点数: 20.40 pts | 救回样本: 102
正在处理: licv_layers_14_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.60% | 后处理后 Acc: 51.00%
提升点数: 20.40 pts | 救回样本: 102
正在处理: licv_layers_15_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.80% | 后处理后 Acc: 56.60%
提升点数: 25.80 pts | 救回样本: 130
正在处理: licv_layers_16_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 28.60% | 后处理后 Acc: 57.20%
提升点数: 28.60 pts | 救回样本: 143
正在处理: licv_layers_17_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 32

gsm8k-mimic-post

In [9]:
import json
import re
import csv
import glob
import os
from fractions import Fraction
from typing import Optional

class MathVistaNormalizer:
    @classmethod
    def to_numeric(cls, raw: str) -> Optional[float]:
        if not raw: return None
        raw = str(raw).replace('$', '').replace(' ', '').lower().strip()
        raw = re.sub(r"(?:cm|m/s|kg|ms|n|grams|degrees|°|%)", "", raw)
        
        # 1. 根式
        sqrt_match = re.search(r"(?:√|\\sqrt\s*\{)\s*(\d+)\s*\}?", raw)
        if sqrt_match: return float(int(sqrt_match.group(1))**0.5)

        # 2. 百分数
        if '%' in raw or r'\%' in raw:
            m = re.search(r"(-?\d+(?:\.\d+)?)", raw)
            if m: return float(m.group(1)) / 100.0

        # 3. 分数
        if '/' in raw:
            m = re.search(r"(-?\d+)\s*/\s*(-?\d+)", raw)
            if m:
                try: return float(Fraction(int(m.group(1)), int(m.group(2))))
                except: pass
            
        # 4. 普通数值提取
        nums = re.findall(r"[-+]?\d*\.?\d+", raw.replace(',', ''))
        if nums:
            try: return float(nums[-1])
            except: return None
        return None

def full_post_process_and_save(results_path, eval_data_path):
    # 加载原始数据索引
    with open(eval_data_path, 'r') as f:
        eval_ref = {str(item['pid']): item for item in json.load(f)}
    
    with open(results_path, 'r') as f:
        data = json.load(f)
    
    records = data['records']
    new_correct = 0
    fix_count = 0
    
    # --- [关键修改]: 定义 log_buffer 和 csv_data ---
    log_buffer = []
    csv_data = []

    for rec in records:
        pid = str(rec['pid'])
        ref = eval_ref.get(pid, {})
        choices = ref.get('choices', [])
        
        ext = str(rec['ext_p']).strip()
        gt = str(rec['gt']).strip()
        is_now_correct = False

        # 1. 选项映射
        if choices and len(ext) == 1 and ext.upper() in "ABCDE":
            idx = ord(ext.upper()) - ord('A')
            if idx < len(choices): ext = str(choices[idx]).strip()

        # 2. 数值对比
        val_p = MathVistaNormalizer.to_numeric(ext)
        val_g = MathVistaNormalizer.to_numeric(gt)
        if val_p is not None and val_g is not None:
            if abs(val_p - val_g) < 1e-4: is_now_correct = True

        # 3. 语义对齐
        if not is_now_correct:
            ext_l = ext.lower().strip().rstrip('.')
            gt_l = gt.lower().strip().rstrip('.')
            bool_map = {"yes": "true", "no": "false", "correct": "true", "incorrect": "false"}
            if bool_map.get(ext_l, ext_l) == bool_map.get(gt_l, gt_l): is_now_correct = True

        # 更新计数
        if is_now_correct:
            new_correct += 1
            if not rec.get('is_correct', False): fix_count += 1
        
        status = "✅" if is_now_correct else "❌"
        if is_now_correct and not rec.get('is_correct', False): status = "⭐ FIX"
        
        # 存入 buffer
        line = f"{pid:<6} | {ext[:25]:<25} | {gt[:20]:<20} | {status}"
        log_buffer.append(line)
        
        # 存入 csv_data 用于后续分析
        csv_data.append([pid, ext, gt, status, rec.get('is_correct', False), is_now_correct])

    total = len(records)
    final_acc = new_correct / total
    
    # --- 1. 打印全部 (Jupyter 会截断，但我们可以看最后几行) ---
    # print("\n".join(log_buffer)) 

    # print("-" * 85)
    print(f"统计汇总: 样本数 {total} | 原始 Acc: {data['eval_result'].get('accuracy', 0):.2%} | 后处理后 Acc: {final_acc:.2%}")
    print(f"提升点数: {(final_acc - data['eval_result'].get('accuracy', 0))*100:.2f} pts | 救回样本: {fix_count}")

    # --- 2. 导出 CSV (推荐用这个看全部 500 条) ---
    csv_name = os.path.basename(results_path).replace(".json", "_analysis.csv")
    with open(csv_name, "w", newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(["PID", "Final_Extracted", "Ground_Truth", "Status", "Original_Correct", "New_Correct"])
        writer.writerows(csv_data)
    
    # print("\n✅ 所有 500 条数据已保存至: mathvista_full_analysis.csv")
    # print(f"\n✅ 所有 500 条数据已保存至: {csv_name}")
    return final_acc, data['eval_result'].get('accuracy', 0)

results_dir = "/data1/wzy/cot-mimic/results/record/mimic_inject_gsm8k"
eval_data_path = "/data1/wzy/cot-mimic/mathvista/data/mathvista_evaluate.json"

# result_files = sorted(glob.glob(os.path.join(results_dir, "licv_layers_*_direct_q_1.0.json")))
result_files = glob.glob(os.path.join(results_dir, "mimic_layers_*_direct_q_1.0.json"))

result_files = sorted(result_files, key=lambda x: int(re.search(r"layers_(\d+)", x).group(1)))


# print(f"发现 {len(result_files)} 个结果文件\n")
summary = []

for path in result_files:
    print("=" * 100)
    print(f"正在处理: {os.path.basename(path)}")
    print("=" * 100)

    # ✅ 接住返回值
    final_acc, orig_acc = full_post_process_and_save(
        results_path=path,
        eval_data_path=eval_data_path
    )

    # ✅ 保存到 summary
    summary.append((os.path.basename(path), orig_acc, final_acc))


# ================= 汇总打印（放在循环外面） =================

# print("\n" + "=" * 90)
# print(f"{'File':40} | {'Orig Acc':9} | {'Post Acc':9} | {'Gain'}")
# print("-" * 90)

best_file, best_acc = None, 0

for name, orig, post in summary:
    gain = (post - orig) * 100
    print(f"{name:40} | {orig*100:8.2f}% | {post*100:8.2f}% | {gain:+.2f}")

    if post > best_acc:
        best_acc = post
        best_file = name

print("-" * 90)
print(f"✅ Best Layer: {best_file} -> {best_acc*100:.2f}%")
print("=" * 90)



正在处理: mimic_layers_0_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.60% | 后处理后 Acc: 52.20%
提升点数: 21.60 pts | 救回样本: 108
正在处理: mimic_layers_1_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 29.20% | 后处理后 Acc: 51.20%
提升点数: 22.00 pts | 救回样本: 110
正在处理: mimic_layers_2_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 31.00% | 后处理后 Acc: 52.60%
提升点数: 21.60 pts | 救回样本: 108
正在处理: mimic_layers_3_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 31.60% | 后处理后 Acc: 52.60%
提升点数: 21.00 pts | 救回样本: 105
正在处理: mimic_layers_4_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 31.20% | 后处理后 Acc: 51.00%
提升点数: 19.80 pts | 救回样本: 99
正在处理: mimic_layers_5_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.40% | 后处理后 Acc: 51.60%
提升点数: 21.20 pts | 救回样本: 106
正在处理: mimic_layers_6_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.40% | 后处理后 Acc: 52.60%
提升点数: 22.20 pts | 救回样本: 111
正在处理: mimic_layers_7_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.80% | 后处理后 Acc: 53.00%
提升点数: 22.20 pts | 救回样本: 111
正在处理: mimic_layers_8_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30

base_inject_gsm8k

In [11]:
import json
import re
import csv
import glob
import os
from fractions import Fraction
from typing import Optional

class MathVistaNormalizer:
    @classmethod
    def to_numeric(cls, raw: str) -> Optional[float]:
        if not raw: return None
        raw = str(raw).replace('$', '').replace(' ', '').lower().strip()
        raw = re.sub(r"(?:cm|m/s|kg|ms|n|grams|degrees|°|%)", "", raw)
        
        # 1. 根式
        sqrt_match = re.search(r"(?:√|\\sqrt\s*\{)\s*(\d+)\s*\}?", raw)
        if sqrt_match: return float(int(sqrt_match.group(1))**0.5)

        # 2. 百分数
        if '%' in raw or r'\%' in raw:
            m = re.search(r"(-?\d+(?:\.\d+)?)", raw)
            if m: return float(m.group(1)) / 100.0

        # 3. 分数
        if '/' in raw:
            m = re.search(r"(-?\d+)\s*/\s*(-?\d+)", raw)
            if m:
                try: return float(Fraction(int(m.group(1)), int(m.group(2))))
                except: pass
            
        # 4. 普通数值提取
        nums = re.findall(r"[-+]?\d*\.?\d+", raw.replace(',', ''))
        if nums:
            try: return float(nums[-1])
            except: return None
        return None

def full_post_process_and_save(results_path, eval_data_path):
    # 加载原始数据索引
    with open(eval_data_path, 'r') as f:
        eval_ref = {str(item['pid']): item for item in json.load(f)}
    
    with open(results_path, 'r') as f:
        data = json.load(f)
    
    records = data['records']
    new_correct = 0
    fix_count = 0
    
    # --- [关键修改]: 定义 log_buffer 和 csv_data ---
    log_buffer = []
    csv_data = []

    for rec in records:
        pid = str(rec['pid'])
        ref = eval_ref.get(pid, {})
        choices = ref.get('choices', [])
        
        ext = str(rec['ext_p']).strip()
        gt = str(rec['gt']).strip()
        is_now_correct = False

        # 1. 选项映射
        if choices and len(ext) == 1 and ext.upper() in "ABCDE":
            idx = ord(ext.upper()) - ord('A')
            if idx < len(choices): ext = str(choices[idx]).strip()

        # 2. 数值对比
        val_p = MathVistaNormalizer.to_numeric(ext)
        val_g = MathVistaNormalizer.to_numeric(gt)
        if val_p is not None and val_g is not None:
            if abs(val_p - val_g) < 1e-4: is_now_correct = True

        # 3. 语义对齐
        if not is_now_correct:
            ext_l = ext.lower().strip().rstrip('.')
            gt_l = gt.lower().strip().rstrip('.')
            bool_map = {"yes": "true", "no": "false", "correct": "true", "incorrect": "false"}
            if bool_map.get(ext_l, ext_l) == bool_map.get(gt_l, gt_l): is_now_correct = True

        # 更新计数
        if is_now_correct:
            new_correct += 1
            if not rec.get('is_correct', False): fix_count += 1
        
        status = "✅" if is_now_correct else "❌"
        if is_now_correct and not rec.get('is_correct', False): status = "⭐ FIX"
        
        # 存入 buffer
        line = f"{pid:<6} | {ext[:25]:<25} | {gt[:20]:<20} | {status}"
        log_buffer.append(line)
        
        # 存入 csv_data 用于后续分析
        csv_data.append([pid, ext, gt, status, rec.get('is_correct', False), is_now_correct])

    total = len(records)
    final_acc = new_correct / total
    
    # --- 1. 打印全部 (Jupyter 会截断，但我们可以看最后几行) ---
    # print("\n".join(log_buffer)) 

    # print("-" * 85)
    print(f"统计汇总: 样本数 {total} | 原始 Acc: {data['eval_result'].get('accuracy', 0):.2%} | 后处理后 Acc: {final_acc:.2%}")
    print(f"提升点数: {(final_acc - data['eval_result'].get('accuracy', 0))*100:.2f} pts | 救回样本: {fix_count}")

    # --- 2. 导出 CSV (推荐用这个看全部 500 条) ---
    csv_name = os.path.basename(results_path).replace(".json", "_analysis.csv")
    with open(csv_name, "w", newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(["PID", "Final_Extracted", "Ground_Truth", "Status", "Original_Correct", "New_Correct"])
        writer.writerows(csv_data)
    
    # print("\n✅ 所有 500 条数据已保存至: mathvista_full_analysis.csv")
    # print(f"\n✅ 所有 500 条数据已保存至: {csv_name}")
    return final_acc, data['eval_result'].get('accuracy', 0)

results_dir = "/data1/wzy/cot-mimic/results/record/base_inject_gsm8k"
eval_data_path = "/data1/wzy/cot-mimic/mathvista/data/mathvista_evaluate.json"

# result_files = sorted(glob.glob(os.path.join(results_dir, "licv_layers_*_direct_q_1.0.json")))
result_files = glob.glob(os.path.join(results_dir, "use_base_vector_layers_*_direct_q_1.0.json"))

result_files = sorted(result_files, key=lambda x: int(re.search(r"layers_(\d+)", x).group(1)))


# print(f"发现 {len(result_files)} 个结果文件\n")
summary = []

for path in result_files:
    print("=" * 100)
    print(f"正在处理: {os.path.basename(path)}")
    print("=" * 100)

    # ✅ 接住返回值
    final_acc, orig_acc = full_post_process_and_save(
        results_path=path,
        eval_data_path=eval_data_path
    )

    # ✅ 保存到 summary
    summary.append((os.path.basename(path), orig_acc, final_acc))


# ================= 汇总打印（放在循环外面） =================

# print("\n" + "=" * 90)
# print(f"{'File':40} | {'Orig Acc':9} | {'Post Acc':9} | {'Gain'}")
# print("-" * 90)

best_file, best_acc = None, 0

for name, orig, post in summary:
    gain = (post - orig) * 100
    print(f"{name:40} | {orig*100:8.2f}% | {post*100:8.2f}% | {gain:+.2f}")

    if post > best_acc:
        best_acc = post
        best_file = name

print("-" * 90)
print(f"✅ Best Layer: {best_file} -> {best_acc*100:.2f}%")
print("=" * 90)



正在处理: use_base_vector_layers_0_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 33.60% | 后处理后 Acc: 52.80%
提升点数: 19.20 pts | 救回样本: 96
正在处理: use_base_vector_layers_1_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 31.00% | 后处理后 Acc: 53.00%
提升点数: 22.00 pts | 救回样本: 110
正在处理: use_base_vector_layers_2_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 32.00% | 后处理后 Acc: 54.60%
提升点数: 22.60 pts | 救回样本: 113
正在处理: use_base_vector_layers_3_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 31.00% | 后处理后 Acc: 60.40%
提升点数: 29.40 pts | 救回样本: 147
正在处理: use_base_vector_layers_4_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.80% | 后处理后 Acc: 53.40%
提升点数: 22.60 pts | 救回样本: 113
正在处理: use_base_vector_layers_5_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 33.60% | 后处理后 Acc: 54.60%
提升点数: 21.00 pts | 救回样本: 105
正在处理: use_base_vector_layers_6_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 34.40% | 后处理后 Acc: 51.20%
提升点数: 16.80 pts | 救回样本: 85
正在处理: use_base_vector_layers_7_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 32.40% | 后处理后 Acc: 56.60%
提升点数: 24.20 pt

mathvista-licv


In [1]:
import json
import re
import csv
import glob
import os
from fractions import Fraction
from typing import Optional

class MathVistaNormalizer:
    @classmethod
    def to_numeric(cls, raw: str) -> Optional[float]:
        if not raw: return None
        raw = str(raw).replace('$', '').replace(' ', '').lower().strip()
        raw = re.sub(r"(?:cm|m/s|kg|ms|n|grams|degrees|°|%)", "", raw)
        
        # 1. 根式
        sqrt_match = re.search(r"(?:√|\\sqrt\s*\{)\s*(\d+)\s*\}?", raw)
        if sqrt_match: return float(int(sqrt_match.group(1))**0.5)

        # 2. 百分数
        if '%' in raw or r'\%' in raw:
            m = re.search(r"(-?\d+(?:\.\d+)?)", raw)
            if m: return float(m.group(1)) / 100.0

        # 3. 分数
        if '/' in raw:
            m = re.search(r"(-?\d+)\s*/\s*(-?\d+)", raw)
            if m:
                try: return float(Fraction(int(m.group(1)), int(m.group(2))))
                except: pass
            
        # 4. 普通数值提取
        nums = re.findall(r"[-+]?\d*\.?\d+", raw.replace(',', ''))
        if nums:
            try: return float(nums[-1])
            except: return None
        return None

def full_post_process_and_save(results_path, eval_data_path):
    # 加载原始数据索引
    with open(eval_data_path, 'r') as f:
        eval_ref = {str(item['pid']): item for item in json.load(f)}
    
    with open(results_path, 'r') as f:
        data = json.load(f)
    
    records = data['records']
    new_correct = 0
    fix_count = 0
    
    # --- [关键修改]: 定义 log_buffer 和 csv_data ---
    log_buffer = []
    csv_data = []

    for rec in records:
        pid = str(rec['pid'])
        ref = eval_ref.get(pid, {})
        choices = ref.get('choices', [])
        
        ext = str(rec['ext_p']).strip()
        gt = str(rec['gt']).strip()
        is_now_correct = False

        # 1. 选项映射
        if choices and len(ext) == 1 and ext.upper() in "ABCDE":
            idx = ord(ext.upper()) - ord('A')
            if idx < len(choices): ext = str(choices[idx]).strip()

        # 2. 数值对比
        val_p = MathVistaNormalizer.to_numeric(ext)
        val_g = MathVistaNormalizer.to_numeric(gt)
        if val_p is not None and val_g is not None:
            if abs(val_p - val_g) < 1e-4: is_now_correct = True

        # 3. 语义对齐
        if not is_now_correct:
            ext_l = ext.lower().strip().rstrip('.')
            gt_l = gt.lower().strip().rstrip('.')
            bool_map = {"yes": "true", "no": "false", "correct": "true", "incorrect": "false"}
            if bool_map.get(ext_l, ext_l) == bool_map.get(gt_l, gt_l): is_now_correct = True

        # 更新计数
        if is_now_correct:
            new_correct += 1
            if not rec.get('is_correct', False): fix_count += 1
        
        status = "✅" if is_now_correct else "❌"
        if is_now_correct and not rec.get('is_correct', False): status = "⭐ FIX"
        
        # 存入 buffer
        line = f"{pid:<6} | {ext[:25]:<25} | {gt[:20]:<20} | {status}"
        log_buffer.append(line)
        
        # 存入 csv_data 用于后续分析
        csv_data.append([pid, ext, gt, status, rec.get('is_correct', False), is_now_correct])

    total = len(records)
    final_acc = new_correct / total
    
    # --- 1. 打印全部 (Jupyter 会截断，但我们可以看最后几行) ---
    # print("\n".join(log_buffer)) 

    # print("-" * 85)
    print(f"统计汇总: 样本数 {total} | 原始 Acc: {data['eval_result'].get('accuracy', 0):.2%} | 后处理后 Acc: {final_acc:.2%}")
    print(f"提升点数: {(final_acc - data['eval_result'].get('accuracy', 0))*100:.2f} pts | 救回样本: {fix_count}")

    # --- 2. 导出 CSV (推荐用这个看全部 500 条) ---
    csv_name = os.path.basename(results_path).replace(".json", "_analysis.csv")
    with open(csv_name, "w", newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(["PID", "Final_Extracted", "Ground_Truth", "Status", "Original_Correct", "New_Correct"])
        writer.writerows(csv_data)
    
    # print("\n✅ 所有 500 条数据已保存至: mathvista_full_analysis.csv")
    # print(f"\n✅ 所有 500 条数据已保存至: {csv_name}")
    return final_acc, data['eval_result'].get('accuracy', 0)

results_dir = "/data1/wzy/cot-mimic/results/record/base_inject_mathvista"
eval_data_path = "/data1/wzy/cot-mimic/mathvista/data/mathvista_evaluate.json"

# result_files = sorted(glob.glob(os.path.join(results_dir, "licv_layers_*_direct_q_1.0.json")))
result_files = glob.glob(os.path.join(results_dir, "use_base_vector_layers_*_direct_q_1.0.json"))

result_files = sorted(result_files, key=lambda x: int(re.search(r"layers_(\d+)", x).group(1)))


# print(f"发现 {len(result_files)} 个结果文件\n")
summary = []

for path in result_files:
    print("=" * 100)
    print(f"正在处理: {os.path.basename(path)}")
    print("=" * 100)

    # ✅ 接住返回值
    final_acc, orig_acc = full_post_process_and_save(
        results_path=path,
        eval_data_path=eval_data_path
    )

    # ✅ 保存到 summary
    summary.append((os.path.basename(path), orig_acc, final_acc))


# ================= 汇总打印（放在循环外面） =================

# print("\n" + "=" * 90)
# print(f"{'File':40} | {'Orig Acc':9} | {'Post Acc':9} | {'Gain'}")
# print("-" * 90)

best_file, best_acc = None, 0

for name, orig, post in summary:
    gain = (post - orig) * 100
    print(f"{name:40} | {orig*100:8.2f}% | {post*100:8.2f}% | {gain:+.2f}")

    if post > best_acc:
        best_acc = post
        best_file = name

print("-" * 90)
print(f"✅ Best Layer: {best_file} -> {best_acc*100:.2f}%")
print("=" * 90)



正在处理: use_base_vector_layers_0_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.60% | 后处理后 Acc: 51.80%
提升点数: 21.20 pts | 救回样本: 106
正在处理: use_base_vector_layers_1_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.80% | 后处理后 Acc: 52.80%
提升点数: 22.00 pts | 救回样本: 110
正在处理: use_base_vector_layers_2_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 32.60% | 后处理后 Acc: 55.80%
提升点数: 23.20 pts | 救回样本: 116
正在处理: use_base_vector_layers_3_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 31.80% | 后处理后 Acc: 56.60%
提升点数: 24.80 pts | 救回样本: 124
正在处理: use_base_vector_layers_4_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 31.60% | 后处理后 Acc: 54.40%
提升点数: 22.80 pts | 救回样本: 114
正在处理: use_base_vector_layers_5_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.00% | 后处理后 Acc: 52.60%
提升点数: 22.60 pts | 救回样本: 113
正在处理: use_base_vector_layers_6_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 31.60% | 后处理后 Acc: 51.60%
提升点数: 20.00 pts | 救回样本: 100
正在处理: use_base_vector_layers_7_direct_q_1.0.json
统计汇总: 样本数 500 | 原始 Acc: 30.40% | 后处理后 Acc: 59.80%
提升点数: 29.40 

In [2]:
import torch

p = "/data1/wzy/cot-mimic/results/average_cot_vectors/old_qwen2.5-vl-7b_gsm8k_avg_cot_vector_mimic.pt"
x = torch.load(
    "/data1/wzy/cot-mimic/results/average_cot_vectors/old_qwen2.5-vl-7b_gsm8k_avg_cot_vector_mimic.pt",
    map_location="cpu",
    weights_only=False)

print(x.keys())
print("ffn_cot_vector shape:", x["ffn_cot_vector"].shape)
print("ffn_hs_vector shape:", x["ffn_hs_vector"].shape)
print("attn_cot_vector shape:", x["attn_cot_vector"].shape if x["attn_cot_vector"] is not None else None)

print("ffn_cot abs mean:", x["ffn_cot_vector"].abs().mean().item())
print("ffn_hs abs mean:", x["ffn_hs_vector"].abs().mean().item())
print("attn_cot abs mean:", x["attn_cot_vector"].abs().mean().item() if x["attn_cot_vector"] is not None else None)

print("ffn_cot has nan:", torch.isnan(x["ffn_cot_vector"]).any().item())
print("ffn_hs has nan:", torch.isnan(x["ffn_hs_vector"]).any().item())
print("attn_cot has nan:", torch.isnan(x["attn_cot_vector"]).any().item() if x["attn_cot_vector"] is not None else None)

dict_keys(['ffn_cot_vector', 'ffn_hs_vector', 'attn_cot_vector', 'encoder_type', 'multi_head_attn_strategy'])
ffn_cot_vector shape: torch.Size([28, 3584])
ffn_hs_vector shape: torch.Size([28, 3584])
attn_cot_vector shape: torch.Size([28, 28, 128])
ffn_cot abs mean: 0.45877277851104736
ffn_hs abs mean: 0.4586361348628998
attn_cot abs mean: 0.11738183349370956
ffn_cot has nan: False
ffn_hs has nan: False
attn_cot has nan: False


In [1]:
import torch

p = "/data1/wzy/cot-mimic/results/average_cot_vectors/qwen2.5-vl-7b_gsm8k_avg_cot_vector_mimic.pt"
x = torch.load(
    "/data1/wzy/cot-mimic/results/average_cot_vectors/qwen2.5-vl-7b_gsm8k_avg_cot_vector_mimic.pt",
    map_location="cpu",
    weights_only=False)

print(x.keys())
print("ffn_cot_vector shape:", x["ffn_cot_vector"].shape)
print("ffn_hs_vector shape:", x["ffn_hs_vector"].shape)
print("attn_cot_vector shape:", x["attn_cot_vector"].shape if x["attn_cot_vector"] is not None else None)

print("ffn_cot abs mean:", x["ffn_cot_vector"].abs().mean().item())
print("ffn_hs abs mean:", x["ffn_hs_vector"].abs().mean().item())
print("attn_cot abs mean:", x["attn_cot_vector"].abs().mean().item() if x["attn_cot_vector"] is not None else None)

print("ffn_cot has nan:", torch.isnan(x["ffn_cot_vector"]).any().item())
print("ffn_hs has nan:", torch.isnan(x["ffn_hs_vector"]).any().item())
print("attn_cot has nan:", torch.isnan(x["attn_cot_vector"]).any().item() if x["attn_cot_vector"] is not None else None)

dict_keys(['ffn_cot_vector', 'ffn_hs_vector', 'attn_cot_vector', 'encoder_type', 'multi_head_attn_strategy'])
ffn_cot_vector shape: torch.Size([28, 3584])
ffn_hs_vector shape: torch.Size([28, 3584])
attn_cot_vector shape: torch.Size([28, 28, 128])
ffn_cot abs mean: 0.3648907542228699
ffn_hs abs mean: 0.4058971107006073
attn_cot abs mean: 0.09686704725027084
ffn_cot has nan: False
ffn_hs has nan: False
attn_cot has nan: False


In [1]:
import torch

p = "/data1/wzy/cot-mimic/results/average_cot_vectors/qwen2.5-vl-7b-instruct_mathvista_correct_only_avg_cot_vector_mimic_fix5_bs1_20260322.pt"
x = torch.load(
    p,
    map_location="cpu",
    weights_only=False)

print(x.keys())
print("ffn_cot_vector shape:", x["ffn_cot_vector"].shape)
print("ffn_hs_vector shape:", x["ffn_hs_vector"].shape)
print("attn_cot_vector shape:", x["attn_cot_vector"].shape if x["attn_cot_vector"] is not None else None)

print("ffn_cot abs mean:", x["ffn_cot_vector"].abs().mean().item())
print("ffn_hs abs mean:", x["ffn_hs_vector"].abs().mean().item())
print("attn_cot abs mean:", x["attn_cot_vector"].abs().mean().item() if x["attn_cot_vector"] is not None else None)

print("ffn_cot has nan:", torch.isnan(x["ffn_cot_vector"]).any().item())
print("ffn_hs has nan:", torch.isnan(x["ffn_hs_vector"]).any().item())
print("attn_cot has nan:", torch.isnan(x["attn_cot_vector"]).any().item() if x["attn_cot_vector"] is not None else None)

dict_keys(['ffn_cot_vector', 'ffn_hs_vector', 'attn_cot_vector', 'encoder_type', 'multi_head_attn_strategy'])
ffn_cot_vector shape: torch.Size([28, 3584])
ffn_hs_vector shape: torch.Size([28, 3584])
attn_cot_vector shape: torch.Size([28, 28, 128])
ffn_cot abs mean: 0.3566419780254364
ffn_hs abs mean: 0.4513654112815857
attn_cot abs mean: 0.15243491530418396
ffn_cot has nan: False
ffn_hs has nan: False
attn_cot has nan: False


In [ ]:
import torch

p = "data1/wzy/cot-mimic/results/average_cot_vectors/qwen2.5-vl-7b_gsm8k_avg_cot_vector_mimic.pt/"
x = torch.load(p, map_location="cpu", weights_only=False)

ffn_cot_norm = x["ffn_cot_vector"].norm(dim=1)
ffn_hs_norm = x["ffn_hs_vector"].norm(dim=1)
attn_cot_norm = x["attn_cot_vector"].flatten(1).norm(dim=1)

print("ffn_cot_norm per layer:")
print(ffn_cot_norm)

print("ffn_hs_norm per layer:")
print(ffn_hs_norm)

print("attn_cot_norm per layer:")
print(attn_cot_norm)

ffn_cot_norm per layer:
tensor([  2.2025,   2.9278,   3.6740,   9.0673,   6.5864,   7.6724,  11.8346,
         11.8355,  13.0600,  13.0301,  14.9612,  15.3707,  17.7070,  20.7976,
         21.4229,  21.9599,  21.2519,  26.1076,  31.6105,  37.7251,  39.3365,
         41.8046,  48.3196,  51.4895,  55.8486,  76.8897, 130.7688, 319.5446])
ffn_hs_norm per layer:
tensor([  4.7093,   4.5573,   6.0011,   9.5915,   7.8175,  10.7561,  12.4699,
         13.4514,  15.1781,  15.5849,  17.6018,  18.0418,  18.1616,  22.8178,
         20.2872,  19.0746,  18.8777,  23.1114,  30.1879,  36.6101,  40.3262,
         43.1262,  47.9723,  58.4952,  63.3439, 107.3162, 140.3585, 417.8827])
attn_cot_norm per layer:
tensor([ 1.6018,  2.4880,  2.2734,  4.0910,  2.2450,  3.7351,  4.4249,  3.0041,
         5.1466,  4.5477,  6.4361,  5.9935,  7.1308,  7.5045,  9.5044,  9.2646,
        10.7553,  9.9561, 13.4547, 15.5831, 11.0993, 17.4675, 12.0477, 15.2945,
        19.2958, 14.3599, 20.9009, 26.2019])
